***
***
# Tutorial illustraing the use of C3S2Q for NaCl
***
***
<br>
In this tutorial we fit a CCS potential alongside a point charge description with effective atomic charges unique to each element. 

<br><br><br><br><br>

***
***
## 1. Import modules

Import the modules needed to generate the training data.

In [ ]:
# Importing modules
from ccs_fit.ase_calculator.ccs_ase_G2B import G2B
from ccs_fit.scripts.ccs_build_db import ccs_build_db
from ccs_fit import ccs_fit
from ccs_fit.scripts.ccs_validate import ccs_validate
from ccs_fit.ase_calculator.ccs_ase_calculator import spline_table
from ccs_fit.ase_calculator.ccs_ase_G2B import G2B_pair

from ase.io import read,write
import ase.db as db
from ase.build import bulk

import numpy as np
import matplotlib.pyplot as plt
import json
import os
import glob

import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm import tqdm


***
***
## 2. Define the reference system and potential


The reference data in the example is generated from an anyltical two-body potential. The potential is specified in a dictionary called `G2B_params`. The analytical form of the two-body potentials are given by `"V_func"` where the function is expressed in terms of the distance using the label `r_ij`. The potential has been shifted such that all 2-body potentials aproach zero at the cutoff.

In [ ]:
# Reference
NaCl=bulk('NaCl','rocksalt',cubic=True,a=5.6)
NaCl=NaCl*[2,2,2]

G2B_params={
        "Charges": {
                "Na": 1.0,
                "Cl": -1.0
        },
        "One_body": {
                "Na": 0.0,
                "Cl": 0.0
        },
        "Two_body": {
                "Na-Na": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.2636971851926334*exp( (2.34- r_ij) /0.317)      -1.0485876722813794/(r_ij**6)-0.4993377877804923/(r_ij**8)+4.025164091303455e-06"
                },
                "Na-Cl": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.21096642097716847*exp( (2.755- r_ij) /0.317)      -6.990512208350348/(r_ij**6)-8.675685093364478/(r_ij**8)+2.7170043029340154e-05"
                },
                "Cl-Cl": {
                        "r_min": 0.0,
                        "r_cut": 8.0,
                        "V_func": " 0.15823565676170354*exp( (3.17- r_ij) /0.317)      -72.40150747359878/(r_ij**6)-145.427198022952/(r_ij**8)+0.00028481976267018326"
                }
        }
}
calc = G2B(G2B_params)
NaCl.calc = calc

***
***
## 3. Generate training data

Curvature Constrained Splines can be fitted to a reference data-set with energies (and optionally forces) of pre-calculated structures. 
> **Note:** Stresses cannot be computed in this example since we use the `pymatgen` library to compute the energy and forces arising from the point charges. However, `pymatgen` does not provide stress. 

We first define a simple python function to generate the training-data.

In [ ]:
# Define training protocol
def do_training():
    orig_cell = NaCl.get_cell()
    orig_struc = NaCl.copy()
    displacement_magnitude=0.05
    disp_steps=4
    scale_steps=10

    counter=1
    total_iterations=scale_steps*disp_steps
    with tqdm(total=total_iterations) as pbar:
        for scale in np.linspace(0.65, 1.25, scale_steps):
            new_cell = orig_cell*scale
            new_struc = orig_struc.copy()
            new_struc.set_cell(new_cell,scale_atoms=True)
            new_struc.calc = calc
            nrg = new_struc.get_potential_energy()
            for i in range(disp_steps):
                rattle_struc = new_struc.copy()
                rattle_struc.rattle(displacement_magnitude*i, seed=counter)
                rattle_struc.calc = calc
                nrg = rattle_struc.get_potential_energy()
                dists = rattle_struc.get_all_distances(mic=True)  
                np.fill_diagonal(dists, np.inf)
                min_d = np.min(dists)
                if min_d > 1.5:
                    xyz_file=f"CALCULATED_DATA/S{counter}.xyz"
                    write(xyz_file,rattle_struc)
                    counter += 1
                pbar.update(1)

       

Next we generate the actual data for our training set. Since the training takes considerable time, you can skip this step and use the pre-computed data instead. 

In [ ]:
# Carry out training
def train(b):
    with out:
        clear_output(wait=True)
        current_directory = os.getcwd()
        for file_path in glob.glob(os.path.join(current_directory, 'CALCULATED_DATA/*')):
            os.remove(file_path)
        do_training()
        print("Training-set completed.")

# Function to cancel cleanup
def cancel_train(b):
    with out:
        clear_output(wait=True)
        print("Skipping generating training-set.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(train)
button_no.on_click(cancel_train)

# Display prompt
print("Would you like to re-create the traning-set? (Note that this is timeconsuming and will overwrite the existing training-set)")

display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)

***
***
## 4. Building a reference database


After generating the training data, we collect it in an ASE database file. The ``file_list`` is a file containing a list of files to be collected into the database.

Example of a ``file_list`` file:

```text
    CALCULATED_DATA/S1.xyz
    CALCULATED_DATA/S2.xyz
    CALCULATED_DATA/S3.xyz
    CALCULATED_DATA/S4.xyz
```
    
Any format supported by ASE can be read in.

In [ ]:
# Write the list of files to a file
f = open("file_list", "w")
current_directory = os.getcwd()

for file_path in glob.glob(os.path.join(current_directory, 'CALCULATED_DATA/*')):
    print(file_path,file=f)
f.close()

With the ``ccs_build_db`` function we can build the data-base from our ``file_list`` like this:

In [ ]:
ccs_build_db(mode="CCS",DFT_DB="NaCl.db",file_list="file_list",overwrite=True)

***
***
## 5. Fit training data to Curvature Constrained Splines + Point charges

We are now ready to fit CCS to the target energies and forces collected in the `NaCl.db`  file. 

> **Note:** Stresses cannot be fitted in this example since we use the `pymatgen` library to compute the energy and forces arising from the point charges. However, `pymatgen` does not provide stress. In order to fit using stress we have to set `"EwaldRoutine" : "lammps"` in the input to `ccs_fit`. This requires you to have `LAMMPS` installed and properly connected to `ASE` since `ccs_fit` make use of the ase lammps calculator.

### Set parameters for the fitting

In [ ]:
# Define fitting
Rcut=8.0
res=0.05

# Generate input.json file
input={
        "General": {
                "Interface": "CCS+Q",
                "FitForces": "True",
                "FitStresses": "False",
                "EwaldRoutine" : "pymatgen",
                "RangeSeparated" : "Stitch",
                "Merging": "True",
        },
        "TrainSet": "NaCl.db",
        "Twobody": {
                "Na-Na": {
                        "Rcut": Rcut,
                        "Resolution": res,
                        "SwType": "rep",
                        "ConstType": "Mono",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                },
                "Na-Cl": {
                        "Rcut": Rcut,
                        "Resolution": res,
                        "SwType": "sw",
                        "ConstType": "Mono",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                },
                "Cl-Cl": {
                        "Rcut": Rcut,
                        "Resolution": res,
                        "SwType": "rep",
                        "ConstType": "Mono",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                }

        },
        "Onebody": [
                "Na",
                "Cl"
        ],
        "Charges": {
                "Na": 1.0,
                "Cl":-1.0
        }
}
# Save file
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)


### Carry out the fitting

In [ ]:
ccs_fit("CCS_input.json")

### Inspect the quality of the fit

In [ ]:
# inspect
CONV=14.39964547842567
method="CCS"

with open(f'{method}_params.json', "r") as f:
    CCS_params = json.load(f)

for pair in CCS_params["Two_body"]:
    #plt.ylim(-0.01,0.01)
    r_cut=CCS_params["Two_body"][pair]["r_cut"]
    r=np.arange(CCS_params["Two_body"][pair]["r_min"], CCS_params["Two_body"][pair]["r_cut"], 0.001)
    elem1, elem2 = pair.split("-")
    q1_ref=G2B_params["Charges"][elem1]
    q2_ref=G2B_params["Charges"][elem2]
    q1=CCS_params["Charges"][elem1]
    q2=CCS_params["Charges"][elem2]
    tb = spline_table(elem1, elem2, CCS_params)
    gtb = G2B_pair(elem1,elem2,G2B_params)
    y1 = [tb.eval_energy(rs)  for rs in r]
    if CCS_params["Range_sep"] == "Stitch":
        y2 = [gtb.eval_energy(rs) + (q1_ref*q2_ref*CONV)/rs  for rs in r]
    else:
        y2 = [gtb.eval_energy(rs) for rs in r]
    plt.plot(r,y2,color='black',label=f"{pair} Reference")
    plt.plot(r,y1,'--',color='red',label=f"{pair} CCS")
    plt.xlabel('Distance (Å)')
    plt.ylabel('Energy (eV)')
    plt.legend()
    plt.show()

err=np.loadtxt(f"{method}_error_energies.out")
err[:,0]=err[:,0]/err[:,3]
err[:,1]=err[:,1]/err[:,3]
plt.xlabel('Reference energy (meV/atom)')
plt.ylabel('Fitted energy (meV/atom)')
plt.plot( [1000*min(err[:,0]),1000*max(err[:,0])],[1000*min(err[:,0]),1000*max(err[:,0])],'--',color='black'  )
plt.scatter(1000*err[:,0],1000*err[:,1],facecolors='none', edgecolors='red')
plt.show()
plt.xlabel('Reference energy (meV/atom)')
plt.ylabel('Error in fit (meV/atom)')
plt.scatter(1000*err[:,0],1000*err[:,2],facecolors='none', edgecolors='red')
plt.show()

try:
    err_F=np.loadtxt(f"{method}_error_forces.out")
    plt.xlabel('Reference force (meV/Å)')
    plt.ylabel('Fitted force (meV/Å)')
    #plt.plot( [min(err_F[:,0]),max(err_F[:,0])],[min(err_F[:,0]),max(err_F[:,0])],'--',color='black')
    #plt.scatter(err_F[:,0],err_F[:,1],facecolors='none', edgecolors='red',alpha=0.1 )
    plt.scatter(1000*err_F[:,0],1000*np.abs(err_F[:,1]-err_F[:,0]),facecolors='none', edgecolors='red',alpha=0.1 )

    plt.show()
except:
    pass


### We can go further and compare force and curvature along the individual 2-boby potentials.

In [ ]:
import numpy as np

params="CCS_params.json"

def compare_potentials(r, V_fit, V_ref, label=""):
    delta_r = r[1] - r[0]  # assumes uniform spacing

    diff = V_fit - V_ref
    ise = np.sum(diff**2) * delta_r
    rmse = np.sqrt(np.mean(diff**2))
    max_err = np.max(np.abs(diff))

    # Force comparison
    F_fit = -np.gradient(V_fit, delta_r)
    F_ref = -np.gradient(V_ref, delta_r)
    force_rmse = np.sqrt(np.mean((F_fit - F_ref)**2))
    #plt.ylim(0.2,0.4)
    plt.xlabel("Distance (Å)")
    plt.ylabel("Force (eV/Å)")
    plt.plot(r[1:-1],F_ref[1:-1],'k-')
    plt.plot(r[1:-1],F_fit[1:-1],'r--')
    plt.show()
    
    C_fit= -np.gradient(F_fit,delta_r)
    C_ref= -np.gradient(F_ref,delta_r)
    #plt.ylim(-10,10)
    plt.xlabel("Distance (Å)")
    plt.ylabel(r"Curvature (eV/Å$^2$)")
    plt.plot(r[2:-2],C_ref[2:-2],'k-')
    plt.plot(r[2:-2],C_fit[2:-2],'r--')
    plt.show()
    
    # Relative error metrics
    energy_range = np.max(V_ref) - np.min(V_ref)
    rel_rmse = rmse / energy_range if energy_range > 0 else np.nan
    nise = ise / (np.sum(V_ref**2) * delta_r) if np.sum(V_ref**2) > 0 else np.nan

    epsilon = 1e-6  # avoid divide-by-zero
    rel_err_array = np.abs(diff) / (np.abs(V_ref) + epsilon)
    max_rel_err = np.max(rel_err_array)

    # Print results
    print(f"Comparison {label}".center(50, "-"))
    print(f"ISE            = {ise:.6e} eV²·Å")
    print(f"RMSE           = {rmse:.6e} eV")
    print(f"Max Abs Error  = {max_err:.6e} eV")
    print(f"Force RMSE     = {force_rmse:.6e} eV/Å")
    print(f"Rel RMSE       = {rel_rmse:.6e} (RMSE / range)")
    print(f"Normalized ISE = {nise:.6e} (ISE / ∫V_ref²)")
    print(f"Max Rel Error  = {max_rel_err:.6e} (with ε = {epsilon})")
    print("-" * 50)

    return {
        "ISE": ise,
        "RMSE": rmse,
        "Max Abs Error": max_err,
        "Force RMSE": force_rmse,
        "Rel RMSE": rel_rmse,
        "Normalized ISE": nise,
        "Max Rel Error": max_rel_err
    }


CONV=14.39964547842567

with open(params, "r") as f:
    CCS_params = json.load(f)

for pair in CCS_params["Two_body"]:
    r_cut=CCS_params["Two_body"][pair]["r_cut"]
    r=np.arange(CCS_params["Two_body"][pair]["r_min"], CCS_params["Two_body"][pair]["r_cut"], 0.001)
    elem1, elem2 = pair.split("-")
    q1_ref=G2B_params["Charges"][elem1]
    q2_ref=G2B_params["Charges"][elem2]
    q1=CCS_params["Charges"][elem1]
    q2=CCS_params["Charges"][elem2]
    tb = spline_table(elem1, elem2, CCS_params)
    gtb = G2B_pair(elem1,elem2,G2B_params)
    V_fitted = np.array([tb.eval_energy(rs)  for rs in r])
    if CCS_params["Range_sep"]=="Stitch":
        V_ref = np.array([gtb.eval_energy(rs) + (q1_ref*q2_ref*CONV)/rs  for rs in r])
    else:
        V_ref = np.array([gtb.eval_energy(rs) for rs in r])
    compare_potentials(r,V_fitted,V_ref,label=pair)

    

### We can also inspect the values of the fitted charges
In this case we know the reference value allowing us to compare.

In [ ]:
for item in CCS_params["Charges"]:
    print(f"Fitted charge of {item} {CCS_params['Charges'][item]} vs target {G2B_params['Charges'][item]}")

***
***
## 6. Validate the potential 

Make sure your potential (at the very least) reproduce the data points in your training-set.

In [ ]:
ccs_validate(mode="CCS",CCS_params="CCS_params.json",DFT_DB="NaCl.db")

err=np.loadtxt(f"{method}_validate.dat")
err[:,0]=err[:,0]/err[:,3]
err[:,1]=err[:,1]/err[:,3]
plt.xlabel('Reference energy (meV/atom)')
plt.ylabel('Error in prediction (meV/atom)')
plt.scatter(1000*err[:,0],1000*err[:,2],facecolors='none', edgecolors='red')
plt.show()

***
***
# 7. Test the potential

We define two calculators for further use. One is the reference potential and the other is the fitted potential.

In [ ]:
# Test the potential
from ccs_fit.ase_calculator.ccs_ase_calculator import CCS
from ccs_fit.ase_calculator.ccs_ase_G2B import G2B
import json

with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

calc_fitted=CCS(CCS_params)
calc_ref=G2B(G2B_params)

### Compare elastic constants to the reference

In [ ]:
# Elastic constants
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from ase import Atoms
from ase.units import GPa

# === Step 1: Set up relaxed structure ===
a = 5.62
atoms = bulk("NaCl", crystalstructure="rocksalt", a=a, cubic=True)

# === Step 2: Define strain matrices ===
strain_values = np.linspace(-0.01, 0.01, 11)  # Small symmetric strains

# For cubic crystals:
strain_patterns = {
    "C11": np.array([[1, 0, 0], [0, 0, 0], [0, 0, 0]]),
    "C12": np.array([[1, 0, 0], [1, 0, 0], [0, 0, 0]]) * 0.5,
    "C44": np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]]) * 0.5
}

results = {}

for label, pattern in strain_patterns.items():
    results[label] = {}

    plt.figure()
    for calc_label, calc in zip(["fitted", "ref"], [calc_fitted, calc_ref]):
        atoms.calc = calc
        volume = atoms.get_volume()

        energies = []
        strains = []

        for eps in strain_values:
            strain = np.eye(3) + eps * pattern
            strained_cell = atoms.get_cell().dot(strain)
            strained_atoms = atoms.copy()
            strained_atoms.set_cell(strained_cell, scale_atoms=True)
            strained_atoms.calc = calc
            e = strained_atoms.get_potential_energy()
            energies.append(e)
            strains.append(eps)

        strains = np.array(strains)
        energies = np.array(energies)
        energies -= energies[np.argmin(np.abs(strains))]  # Zero at zero strain

        coeff = np.polyfit(strains, energies, 2)[0]
        C_ij = (2 * coeff) / volume / GPa
        results[label][calc_label] = (strains, energies * 1000, C_ij)  # energy in meV

        
        
        plt.plot(strains, energies * 1000, 'o-', label=f"{calc_label} model")
    plt.xlabel("Strain")
    plt.ylabel("Energy (meV)")
    plt.title(f"{label}: Energy vs Strain")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# === Step 3: Print comparison table ===
print("\nElastic constants (GPa):")
print(f"{'Label':<5} {'Fitted':>10} {'Reference':>12} {'Diff':>10}")
for key in strain_patterns:
    C_fit = results[key]['fitted'][2]
    C_ref = results[key]['ref'][2]
    diff = C_fit - C_ref
    print(f"{key:<5} {C_fit:10.2f} {C_ref:12.2f} {diff:10.2f}")


### Compare phonon spectra to the reference

In [ ]:
# Phonons
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from ase import Atoms
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms

# === Step 1: Build primitive NaCl structure ===
a = 5.62
atoms = bulk("NaCl", crystalstructure="rocksalt", a=a)

def ase_to_phonopy_atoms(atoms):
    return PhonopyAtoms(
        symbols=atoms.get_chemical_symbols(),
        positions=atoms.get_positions(),
        cell=atoms.get_cell(),
    )

unitcell = ase_to_phonopy_atoms(atoms)

# === Step 2: Set up phonon object ===
supercell_matrix = [[3, 0, 0], [0, 3, 0], [0, 0, 3]]
phonon_fitted = Phonopy(unitcell, supercell_matrix)
phonon_ref = Phonopy(unitcell, supercell_matrix)

phonon_fitted.generate_displacements(distance=0.01)
phonon_ref.generate_displacements(distance=0.01)

supercells_fitted = phonon_fitted.get_supercells_with_displacements()
supercells_ref = phonon_ref.get_supercells_with_displacements()

# === Step 3: Evaluate forces ===
def get_forces(supercells, calculator):
    forces = []
    for sc in supercells:
        atoms_sc = Atoms(
            symbols=sc.get_chemical_symbols(),
            positions=sc.get_positions(),
            cell=sc.get_cell(),
            pbc=True
        )
        atoms_sc.calc = calculator
        forces.append(atoms_sc.get_forces())
    return forces

forces_fitted = get_forces(supercells_fitted, calc_fitted)
forces_ref = get_forces(supercells_ref, calc_ref)

# === Step 4: Store forces and compute force constants ===
phonon_fitted.set_forces(forces_fitted)
phonon_ref.set_forces(forces_ref)

phonon_fitted.produce_force_constants()
phonon_ref.produce_force_constants()

# === Step 5: Define FCC band path manually ===
band_points = {
    "Γ": [0.0, 0.0, 0.0],
    "X": [0.5, 0.0, 0.0],
    "W": [0.5, 0.25, 0.0],
    "K": [0.375, 0.375, 0.0],
    "L": [0.5, 0.5, 0.5],
    "U": [0.625, 0.25, 0.625],
}

path_sequence = ["Γ", "X", "W", "K", "Γ", "L", "U", "W"]
npoints = 101

bands = []
tick_positions = []
tick_labels = []

x = []
distance = 0.0
tick_positions.append(0.0)
tick_labels.append(r"$\Gamma$")

reciprocal_lattice = np.linalg.inv(phonon_fitted.primitive.cell).T

for i in range(len(path_sequence) - 1):
    start = np.array(band_points[path_sequence[i]])
    end = np.array(band_points[path_sequence[i + 1]])
    segment = [start + (end - start) * t / (npoints - 1) for t in range(npoints)]
    bands.append(segment)

    dk = np.linalg.norm(np.dot(end - start, reciprocal_lattice))
    distance += dk
    label = r"$\Gamma$" if path_sequence[i + 1] == "Γ" else f"${path_sequence[i + 1]}$"
    tick_positions.append(distance)
    tick_labels.append(label)

flat_bands = [q for band in bands for q in band]

# === Step 6: Compute frequencies ===
phonon_fitted.set_band_structure([flat_bands])
phonon_ref.set_band_structure([flat_bands])
frequencies_fitted = phonon_fitted.get_band_structure_dict()['frequencies'][0]
frequencies_ref  = phonon_ref.get_band_structure_dict()['frequencies'][0]

# === Step 7: Build x-axis ===
x = [0.0]
for i in range(1, len(flat_bands)):
    dq = np.array(flat_bands[i]) - np.array(flat_bands[i - 1])
    dx = np.linalg.norm(np.dot(dq, reciprocal_lattice))
    x.append(x[-1] + dx)
x = np.array(x)

# === Step 8: Convert to cm^-1 ===
THZ_TO_CM1 = 33.35641
frequencies_fitted = np.array(frequencies_fitted) * THZ_TO_CM1
frequencies_ref = np.array(frequencies_ref) * THZ_TO_CM1

# === Step 9: Quantitative comparison ===
mse = np.mean((frequencies_fitted - frequencies_ref)**2)
rmsd = np.sqrt(mse)
mae = np.mean(np.abs(frequencies_fitted - frequencies_ref))
print(f"\nPhonon comparison metrics:")
print(f"RMSD: {rmsd:.2f} cm^-1")
print(f"MAE : {mae:.2f} cm^-1")

# === Step 10: Vibrational free energy and Cv comparison ===
mesh = [30, 30, 30]
phonon_fitted.set_mesh(mesh)
phonon_fitted.set_thermal_properties()
phonon_ref.set_mesh(mesh)
phonon_ref.set_thermal_properties()

temperatures = phonon_fitted.get_thermal_properties_dict()['temperatures']
fitted_fvib = np.array(phonon_fitted.get_thermal_properties_dict()['free_energy'])
ref_fvib = np.array(phonon_ref.get_thermal_properties_dict()['free_energy'])
fitted_cv = np.array(phonon_fitted.get_thermal_properties_dict()['heat_capacity'])
ref_cv = np.array(phonon_ref.get_thermal_properties_dict()['heat_capacity'])

# Free energy plot
plt.figure()
plt.plot(temperatures, ref_fvib, 'k-', label='Reference')
plt.plot(temperatures, fitted_fvib, 'r--', label='Fitted')
plt.xlabel("Temperature (K)")
plt.ylabel("Free energy (eV)")
plt.title("Vibrational Free Energy vs Temperature")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Cv plot
plt.figure()
plt.plot(temperatures, ref_cv, 'k-', label='Reference')
plt.plot(temperatures, fitted_cv, 'r--', label='Fitted')
plt.xlabel("Temperature (K)")
plt.ylabel("Heat Capacity $C_v$ (kB/unit cell)")
plt.title("Vibrational Heat Capacity vs Temperature")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# === Step 11: Plot phonon band structure ===
plt.rcParams['font.family'] = 'serif'
plt.figure(figsize=(6, 5))
num_modes = frequencies_fitted.shape[1]

for i in range(num_modes):
    plt.plot(x, frequencies_ref[:, i], 'k-', linewidth=2)
    plt.plot(x, frequencies_fitted[:, i], 'r--', linewidth=2)

for pos in tick_positions:
    plt.axvline(pos, color='black', linestyle='-', linewidth=1)

plt.xticks(tick_positions, tick_labels, fontsize=13)
plt.ylabel("Frequency (cm$^{-1}$)", fontsize=13)
plt.xlim(x[0], x[-1])
plt.ylim(0, None)
plt.title("Phonon Band Structure", loc='left', fontsize=13)
plt.legend(['Reference', 'Fitted'], fontsize=11)
plt.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.tight_layout()
plt.show()


***
***
## 8. Export potential

After checking the quality of our fitted potential we can export it for use with e.g. lammps.

You can for example export the potential to the *uf3*-format or *table*-format using the following commands. A specification to be used in the lammps input will be printed to screen and the parameters will be saved to a file called `CCS.uf3`.

In [ ]:
#export
from ccs_fit.scripts.ccs_export_FF import ccs_export_FF
print("--- specification for UF3 ---")
print("")
ccs_export_FF("CCS_params.json",form="lammps_uf3")
print("")
print("")
print("--- specification for table ---")
print("")
ccs_export_FF("CCS_params.json",form="lammps_table")

***
***
## 9. Reset notebook 


Use the following box to reset the notebook.

In [ ]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the directories
current_directory = os.getcwd()

# Function to remove files except for the specified file endings
def remove_files_except(directory, exception_endings):
    for file_path in glob.glob(os.path.join(directory, '*')):
        if os.path.isfile(file_path) and not any(file_path.endswith(ending) for ending in exception_endings):
            os.remove(file_path)


# Cleanup function triggered by button click
def cleanup(b):
    with out:
        clear_output(wait=True)
        remove_files_except(current_directory, ['Tutorial.ipynb'])
        remove_files_except(current_directory+'/phonon', [])
        print("Cleanup completed.")

# Function to cancel cleanup
def cancel_cleanup(b):
    with out:
        clear_output(wait=True)
        print("No changes made.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(cleanup)
button_no.on_click(cancel_cleanup)

# Display prompt
print("Would you like to clean up?")
display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)
